# Practical 5: Ensemble Learning - Random Forest on Car Evaluation

Predict the acceptability class of a car (`unacc`, `acc`, `good`, `vgood`) from its categorical attributes.

**Dataset:** `car_evaluation.csv` (Kaggle "Car Evaluation Data Set"). On Colab keep it in `MyDrive/Data-Science`; on local Jupyter keep it next to this notebook.

In [ ]:
# Where the dataset lives:
#  - Google Colab: mounts Google Drive and reads from MyDrive/Data-Science
#  - Local Jupyter: put the CSV in the same folder as this notebook
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/Data-Science'
except ImportError:
    DATA_DIR = '.'

In [ ]:
import pandas as pd

CSV_PATH = os.path.join(DATA_DIR, "car_evaluation.csv")
if not os.path.exists(CSV_PATH):
    # Public UCI copy of the same file (also has no header row)
    CSV_PATH = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

df = pd.read_csv(CSV_PATH)
df.head()

The CSV has no header row, so the first read treats the first record as column names. Re-read it with explicit names:

In [ ]:
column_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv(CSV_PATH, names=column_names, header=None)
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Separate features (X) and target (y)
X = df.drop('class', axis=1)
y = df['class']

# Identify categorical columns (all columns are categorical in this dataset except the target)
categorical_features = X.columns

# Apply One-Hot Encoding to categorical features
# Use ColumnTransformer for robust encoding
preprocessor = ColumnTransformer(
    transformers=[('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)]
)
X_encoded = preprocessor.fit_transform(X)

# Encode the target variable (y) using LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

In [ ]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))